In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\adint\AppData\Local\Temp\ipykernel_30804\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\adint\Downloads\ragpipe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: 2601.03329v1.pdf
  ✓ Loaded 44 pages

Total documents loaded: 44


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\2601.03329v1.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': '2601.03329v1.pdf', 'file_type': 'pdf'}, page_content='Q\nKV\nsoftmax\nα ij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.485

embedding And vectorStoreDB

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
import numpy as np
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    """Handles document embedding genreation using senetencetransformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """initialize the embedding manager
        Args:model_name : Hugginface model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading model {self.model_name}...")
            self.model=SentenceTransformer(self.model_name)
            print(f"✅ Model {self.model_name} loaded successfully!,embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"❌ Error loading model {self.model_name}: {e}")
            raise e 
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

         Args:
        texts: List of text strings to embed

        Returns:
        numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings
    def get_embedding_dimension(self) -> int:
        """Get the dimension of the embeddings"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    


##initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading model all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4808.32it/s]


✅ Model all-MiniLM-L6-v2 loaded successfully!,embedding dimension: 384


C:\Users\adint\AppData\Local\Temp\ipykernel_30804\2144823268.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model {self.model_name} loaded successfully!,embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Text splitting get into chunks

In [6]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 44 documents into 132 chunks

Example chunk:
Content: Q
KV
softmax
α ij
Attention Mechanisms
in Neural Networks
A Comprehensive Mathematical Treatment
From Theory to Implementation
Hasi Hays
arXiv:2601.03329v1  [cs.LG]  6 Jan 2026...
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\2601.03329v1.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': '2601.03329v1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\2601.03329v1.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': '2601.03329v1.pdf', 'file_type': 'pdf'}, page_content='Q\nKV\nsoftmax\nα ij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.485

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection with cosine distance"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Delete existing collection if it exists
            try:
                self.client.delete_collection(name=self.collection_name)
                print(f"✅ Deleted existing collection: {self.collection_name}")
            except:
                pass  # Collection doesn't exist yet, which is fine
            
            # Create new collection with cosine distance metric
            self.collection = self.client.create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"
                }
            )
            print(f"✅ Vector store initialized with cosine distance. Collection: {self.collection_name}")
            print(f"   Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"❌ Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"✅ Successfully added {len(documents)} documents to vector store")
            print(f"   Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"❌ Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

✅ Deleted existing collection: pdf_documents
✅ Vector store initialized with cosine distance. Collection: pdf_documents
   Existing documents in collection: 0


In [9]:
chunks



[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\2601.03329v1.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': '2601.03329v1.pdf', 'file_type': 'pdf'}, page_content='Q\nKV\nsoftmax\nα ij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.485

In [10]:
##convert the text of chunks to embeddings 

### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts



['Q\nKV\nsoftmax\nα ij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026',
 'Abstract\nAttention mechanisms represent a fundamental paradigm shift in neural network architectures, enabling\nmodels to selectively focus on relevant portions of input sequences through learned weighting functions.\nThis monograph provides a comprehensive and rigorous mathematical treatment of attention mechanisms,\nencompassing their theoretical foundations, computational properties, and practical implementations in\ncontemporary deep learning systems. We begin by establishing the mathematical framework for attention\noperations, defining the core components of queries, keys, and values, and analyzing various scoring functions\nincluding additive, multiplicative, and scaled dot-product attention. The permutation equivariance prop-\nerty of self-attention is proven, and its implications for pos

In [11]:
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 132 texts...


Batches: 100%|██████████| 5/5 [00:06<00:00,  1.21s/it]


Generated embeddings with shape: (132, 384)
Adding 132 documents to vector store...
✅ Successfully added 132 documents to vector store
   Total documents in collection: 132


In [12]:
#Retriever pipeline from vectorestore

#RAG RETRIEVER

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query using cosine similarity
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold (0-1, where 1=identical)
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"\n🔍 Retrieving documents for query: '{query}'")
        print(f"   Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Check vector store status
        total_docs = self.vector_store.collection.count()
        print(f"   Documents in store: {total_docs}")
        
        if total_docs == 0:
            print("   ⚠️  Vector store is empty!")
            return []
        
        # Generate query embedding
        print(f"   Generating query embedding...")
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                print(f"\n   📊 Similarity Scores (cosine distance):")
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # With cosine distance: 0 = most similar, 2 = least similar
                    # Similarity score = 1 - distance (normalized to 0-1 range where 1=identical)
                    similarity_score = 1 - distance
                    
                    status = "✅" if similarity_score >= score_threshold else "❌"
                    print(f"      [{i+1}] {status} Distance: {distance:.4f} → Similarity: {similarity_score:.4f}")
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"\n   ✅ Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("   ⚠️  No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"   ❌ Error during retrieval: {e}")
            import traceback
            traceback.print_exc()
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriever

In [15]:

rag_retriever.retrieve("What are The dominant sequence transduction models  based on ")


🔍 Retrieving documents for query: 'What are The dominant sequence transduction models  based on '
   Top K: 5, Score threshold: 0.0
   Documents in store: 132
   Generating query embedding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.94it/s]

Generated embeddings with shape: (1, 384)

   📊 Similarity Scores (cosine distance):
      [1] ✅ Distance: 0.6526 → Similarity: 0.3474
      [2] ✅ Distance: 0.6586 → Similarity: 0.3414
      [3] ✅ Distance: 0.6616 → Similarity: 0.3384
      [4] ✅ Distance: 0.6620 → Similarity: 0.3380
      [5] ✅ Distance: 0.6745 → Similarity: 0.3255

   ✅ Retrieved 5 documents (after filtering)


[{'id': 'doc_ecca5f24_126',
  'content': '[15] Albert Gu, Karan Goel, and Christopher R´ e. Ef-\nficiently modeling long sequences with structured\nstate spaces. InInternational Conference on Learn-\ning Representations, 2022.\n[16] Hasi Hays, Yue Yu, and William J Richardson. Hier-\narchical molecular language models (hmlms).arXiv\npreprint arXiv:2512.00696, 2025.\n[17] Kaiming He, Xiangyu Zhang, Shaoqing Ren, and\nJian Sun. Deep residual learning for image recog-\nnition. InProceedings of the IEEE Conference on\nComputer Vision and Pattern Recognition, pages\n770–778, 2016.\n[18] Jiri Hron, Yasaman Bahri, Jascha Sohl-Dickstein,\nand Roman Novak. Infinite attention: Nngp and ntk\nfor deep attention networks.International Confer-\nence on Machine Learning, pages 4376–4386, 2020.\n[19] Sarthak Jain and Byron C Wallace. Attention is\nnot explanation. InProceedings of the 2019 Con-\nference of the North American Chapter of the Asso-\nciation for Computational Linguistics, pages 3543–\n355

In [16]:
print(vectorstore.collection.count())

132


In [17]:
print(vectorstore.collection.metadata)

{'description': 'PDF document embeddings for RAG', 'hnsw:space': 'cosine'}


Integration Vectordb Context Pieline with LLM output 

In [18]:
#simple rag pipeline with groq llm

import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

##init the groq llm
groq_api_key=os.getenv("GROQ_API_KEY")

In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [20]:
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

In [21]:
## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [22]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)


🔍 Retrieving documents for query: 'What is attention mechanism?'
   Top K: 3, Score threshold: 0.0
   Documents in store: 132
   Generating query embedding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.90it/s]

Generated embeddings with shape: (1, 384)

   📊 Similarity Scores (cosine distance):
      [1] ✅ Distance: 0.3370 → Similarity: 0.6630
      [2] ✅ Distance: 0.3372 → Similarity: 0.6628
      [3] ✅ Distance: 0.3466 → Similarity: 0.6534

   ✅ Retrieved 3 documents (after filtering)


An attention mechanism is a differentiable function that computes a weighted combination of values based on the compatibility between a query and keys.


In [26]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is attention is all you need?", rag_retriever, llm, top_k=8, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


🔍 Retrieving documents for query: 'What is attention is all you need?'
   Top K: 8, Score threshold: 0.1
   Documents in store: 132
   Generating query embedding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.40it/s]

Generated embeddings with shape: (1, 384)

   📊 Similarity Scores (cosine distance):
      [1] ✅ Distance: 0.4639 → Similarity: 0.5361
      [2] ✅ Distance: 0.4753 → Similarity: 0.5247
      [3] ✅ Distance: 0.5128 → Similarity: 0.4872
      [4] ✅ Distance: 0.5138 → Similarity: 0.4862
      [5] ✅ Distance: 0.5155 → Similarity: 0.4845
      [6] ✅ Distance: 0.5226 → Similarity: 0.4774
      [7] ✅ Distance: 0.5292 → Similarity: 0.4708
      [8] ✅ Distance: 0.5372 → Similarity: 0.4628

   ✅ Retrieved 8 documents (after filtering)


Answer: "Attention is All You Need" is the title of a research paper published in 2017 by Vaswani et al. It introduced the Transformer architecture, which revolutionized the field of natural language processing by using self-attention mechanisms to process input sequences. The paper's title is a play on the phrase "Attention is All You Need" from the song "All You Need is Love" by The Beatles.
Sources: [{'source': '2601.03329v1.pdf', 'page': 10, 'score': 0.5360760688781738, 'preview': 'constraints.\n2.Efficiency: Computation should be parallelizable across positions to leverage modern hardware.\n3.Adaptivity: The patterns of information flow should be learned from data rather than hardcoded.\n4.Bounded Memory: Memory requirements should not grow excessively with sequence length.\n5.Lo...'}, {'source': '2601.03329v1.pdf', 'page': 37, 'score': 0.5246981978416443, 'preview': 'in this monograph establish the theoretical basis for understanding these mechanisms, while highlighting\nopen que

In [27]:
# Example usage:
result = rag_advanced("What is self attention?", rag_retriever, llm, top_k=8, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


🔍 Retrieving documents for query: 'What is self attention?'
   Top K: 8, Score threshold: 0.1
   Documents in store: 132
   Generating query embedding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.12it/s]

Generated embeddings with shape: (1, 384)

   📊 Similarity Scores (cosine distance):
      [1] ✅ Distance: 0.4569 → Similarity: 0.5431
      [2] ✅ Distance: 0.4573 → Similarity: 0.5427
      [3] ✅ Distance: 0.4655 → Similarity: 0.5345
      [4] ✅ Distance: 0.4663 → Similarity: 0.5337
      [5] ✅ Distance: 0.4872 → Similarity: 0.5128
      [6] ✅ Distance: 0.4987 → Similarity: 0.5013
      [7] ✅ Distance: 0.4987 → Similarity: 0.5013
      [8] ✅ Distance: 0.5206 → Similarity: 0.4794

   ✅ Retrieved 8 documents (after filtering)


Answer: Self-attention is a mechanism in which a sequence attends to itself, allowing each position to gather information from all positions in the same sequence. It computes queries, keys, and values through learned linear projections, and then computes attention weights to determine how much each position attends to other positions. The output is a matrix where each row contains a contextualized representation of a position, aggregating information from all positions according to learned attention weights.
Sources: [{'source': '2601.03329v1.pdf', 'page': 19, 'score': 0.5431117415428162, 'preview': 'the first position or special tokens) receive attention from most other positions. These positions act as\ninformation aggregators, collecting global information that is then broadcast to other positions in subsequent\nlayers.\n3.3.2 Attention Entropy\nThe entropy of attention distribution for positioni...'}, {'source': '2601.03329v1.pdf', 'page': 16, 'score': 0.5426644086837769, 'preview'